# Feature Target Analysis

Notebook này chỉ đọc các bảng clean/gold đã tạo từ `01_data_processing.ipynb` và phân tích quan hệ giữa feature với target. Mục tiêu là tìm tín hiệu thật sự hữu ích trước khi chọn feature/model cho local app.

Các nhóm phân tích chính:

- Correlation tuyến tính và đơn điệu: Pearson, Spearman.
- Tín hiệu phi tuyến: mutual information, univariate tree lift.
- Feature importance đa biến: permutation importance trên baseline linear/tree model.
- Long-term risk/production: active, availability, games, MPG, per-36 production.

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import brier_score_loss, mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

## Load Clean Data

Cell này tự tìm data trong Drive khi chạy Colab hoặc dùng local `data/` khi chạy trong repo.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
COLAB_DATA_DIR = Path("/content/drive/MyDrive/nba-scout-assistant/data")
LOCAL_DATA_DIR = PROJECT_ROOT / "data"


def first_existing_path(paths: list[Path]) -> Path:
    """Input: candidate data roots. Output: first path that exists, otherwise local data path."""
    for path in paths:
        if path.exists():
            return path
    return LOCAL_DATA_DIR


DATA_DIR = Path(os.getenv("NBA_SCOUT_DATA_DIR", "")).expanduser() if os.getenv("NBA_SCOUT_DATA_DIR") else first_existing_path([COLAB_DATA_DIR, LOCAL_DATA_DIR])
GOLD_DIR = DATA_DIR / "gold"

PERFORMANCE_PATH = GOLD_DIR / "performance_training_clean.parquet"
ROLE_PATH = GOLD_DIR / "player_role_features_clean.parquet"
SALARY_PATH = GOLD_DIR / "salary_training_clean.parquet"
LONG_TERM_PATH = GOLD_DIR / "long_term_player_forecast_training.parquet"

print("DATA_DIR:", DATA_DIR)
for path in [PERFORMANCE_PATH, ROLE_PATH, SALARY_PATH, LONG_TERM_PATH]:
    print(path.name, "exists=", path.exists())

performance = pd.read_parquet(PERFORMANCE_PATH)
role_features = pd.read_parquet(ROLE_PATH)
salary = pd.read_parquet(SALARY_PATH)
long_term = pd.read_parquet(LONG_TERM_PATH)

print("performance", performance.shape)
print("role_features", role_features.shape)
print("salary", salary.shape)
print("long_term", long_term.shape)

## Shared Analysis Helpers

Các helper dưới đây cố tình trả ra bảng gọn để bạn copy output đem qua thảo luận. `split=train` được ưu tiên cho feature discovery để giảm bias từ việc nhìn vào test/final holdout.

In [ ]:
METADATA_HINTS = {
    "player_id", "player_name", "game_id", "as_of_date", "game_date", "season", "team_id", "team", "opponent", "home_away", "split",
    "source", "source_file", "collected_at", "anchor_season", "anchor_date", "anchor_season_start_year",
}


def infer_feature_columns(df: pd.DataFrame, target_cols: list[str], extra_exclude: set[str] | None = None) -> list[str]:
    """Input: dataframe and targets. Output: candidate feature columns after removing metadata and future targets."""
    extra_exclude = extra_exclude or set()
    excluded = set(target_cols) | METADATA_HINTS | extra_exclude
    future_like_prefixes = (
        "target_", "active_h", "games_played_h", "minutes_per_game_h", "pts_per_36_h", "ast_per_36_h", "reb_per_36_h",
        "low_availability_h", "high_availability_h",
    )
    return [
        column for column in df.columns
        if column not in excluded and not any(column.startswith(prefix) for prefix in future_like_prefixes)
    ]


def split_for_analysis(df: pd.DataFrame, preferred_split: str = "train") -> pd.DataFrame:
    """Input: dataframe with optional split column. Output: preferred split rows, falling back to full dataframe."""
    if "split" in df.columns and df["split"].eq(preferred_split).any():
        return df[df["split"].eq(preferred_split)].copy()
    return df.copy()


def numeric_feature_columns(df: pd.DataFrame, features: list[str]) -> list[str]:
    """Input: dataframe and feature list. Output: numeric feature subset."""
    return [column for column in features if column in df.columns and pd.api.types.is_numeric_dtype(df[column])]


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """Input: raw feature matrix. Output: preprocessing transformer for numeric and categorical columns."""
    numeric_cols = [column for column in X.columns if pd.api.types.is_numeric_dtype(X[column])]
    categorical_cols = [column for column in X.columns if column not in numeric_cols]
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([("impute", SimpleImputer(strategy="median"))]), numeric_cols),
            ("categorical", Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), categorical_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def sample_rows(df: pd.DataFrame, max_rows: int = 60000, random_state: int = 42) -> pd.DataFrame:
    """Input: dataframe. Output: sampled dataframe to keep expensive analysis fast and repeatable."""
    if len(df) <= max_rows:
        return df.copy()
    return df.sample(max_rows, random_state=random_state)

## Correlation And Mutual Information

- Pearson: tốt cho quan hệ tuyến tính.
- Spearman: tốt cho quan hệ đơn điệu nhưng không nhất thiết tuyến tính.
- Mutual information: bắt được quan hệ phi tuyến, threshold, interaction đơn giản.

Gợi ý đọc nhanh:

- Pearson cao và Spearman cao: feature khá tuyến tính/đơn điệu với target.
- Pearson thấp nhưng MI cao: có thể là phi tuyến hoặc chỉ hữu ích ở một vùng nhất định.
- Pearson/Spearman trái dấu hoặc lệch mạnh: nên vẽ plot để kiểm tra outlier/threshold.

In [ ]:
def correlation_summary(df: pd.DataFrame, target_col: str, feature_cols: list[str], top_n: int = 30) -> pd.DataFrame:
    """Input: dataframe, target, features. Output: Pearson/Spearman correlation ranking for numeric features."""
    numeric_cols = numeric_feature_columns(df, feature_cols)
    rows = []
    target = pd.to_numeric(df[target_col], errors="coerce")
    for feature in numeric_cols:
        feature_values = pd.to_numeric(df[feature], errors="coerce")
        pair = pd.concat([feature_values, target], axis=1).dropna()
        if len(pair) < 100 or pair.iloc[:, 0].nunique() <= 1 or pair.iloc[:, 1].nunique() <= 1:
            continue
        pearson = pair.iloc[:, 0].corr(pair.iloc[:, 1], method="pearson")
        spearman = pair.iloc[:, 0].corr(pair.iloc[:, 1], method="spearman")
        rows.append({
            "target": target_col,
            "feature": feature,
            "rows": len(pair),
            "pearson": pearson,
            "spearman": spearman,
            "abs_pearson": abs(pearson),
            "abs_spearman": abs(spearman),
            "monotonic_minus_linear": abs(spearman) - abs(pearson),
        })
    result = pd.DataFrame(rows)
    if result.empty:
        return result
    return result.sort_values(["abs_spearman", "abs_pearson"], ascending=False).head(top_n).reset_index(drop=True)


def mutual_information_summary(
    df: pd.DataFrame,
    target_col: str,
    feature_cols: list[str],
    task_type: str = "regression",
    top_n: int = 30,
    max_rows: int = 60000,
) -> pd.DataFrame:
    """Input: dataframe, target, features, task type. Output: mutual-information ranking grouped back to raw features."""
    model_df = sample_rows(df.dropna(subset=[target_col]), max_rows=max_rows)
    X = model_df[feature_cols].copy()
    y = model_df[target_col]
    preprocessor = build_preprocessor(X)
    X_transformed = preprocessor.fit_transform(X)
    encoded_names = list(preprocessor.get_feature_names_out())

    if task_type == "classification":
        scores = mutual_info_classif(X_transformed, y.astype(int), random_state=42, discrete_features="auto")
    else:
        scores = mutual_info_regression(X_transformed, pd.to_numeric(y, errors="coerce"), random_state=42)

    categorical_cols = [column for column in X.columns if not pd.api.types.is_numeric_dtype(X[column])]

    def raw_feature_name(encoded_name: str) -> str:
        """Input: encoded feature name. Output: original raw feature name."""
        if encoded_name in feature_cols:
            return encoded_name
        for column in sorted(categorical_cols, key=len, reverse=True):
            if encoded_name.startswith(f"{column}_"):
                return column
        return encoded_name

    result = pd.DataFrame({"encoded_feature": encoded_names, "mutual_info": scores})
    result["feature"] = result["encoded_feature"].map(raw_feature_name)
    grouped = result.groupby("feature", as_index=False)["mutual_info"].sum()
    grouped["target"] = target_col
    return grouped[["target", "feature", "mutual_info"]].sort_values("mutual_info", ascending=False).head(top_n).reset_index(drop=True)


## Nonlinearity Scan

So sánh một feature đơn lẻ qua 2 model cực nhỏ:

- Ridge/logistic chỉ bắt tín hiệu tuyến tính.
- HistGradientBoosting bắt tín hiệu cong/threshold.

Nếu tree tốt hơn linear nhiều, feature đó có thể có quan hệ phi tuyến với target.

In [ ]:
def univariate_signal_scan(
    df: pd.DataFrame,
    target_col: str,
    feature_cols: list[str],
    task_type: str = "regression",
    top_n: int = 30,
    max_features: int = 80,
    max_rows: int = 50000,
) -> pd.DataFrame:
    """Input: dataframe, target, features, task type. Output: univariate linear-vs-tree score comparison."""
    numeric_cols = numeric_feature_columns(df, feature_cols)[:max_features]
    model_df = sample_rows(df.dropna(subset=[target_col]), max_rows=max_rows)
    rows = []
    for feature in numeric_cols:
        pair = model_df[[feature, target_col]].dropna()
        if len(pair) < 500 or pair[feature].nunique() <= 5 or pair[target_col].nunique() <= 1:
            continue
        X = pair[[feature]]
        y = pair[target_col]
        X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42)
        if task_type == "classification":
            if y_train.nunique() < 2 or y_valid.nunique() < 2:
                continue
            linear = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
            tree = Pipeline([("impute", SimpleImputer(strategy="median")), ("model", HistGradientBoostingClassifier(max_iter=80, learning_rate=0.05, random_state=42))])
            linear.fit(X_train, y_train.astype(int))
            tree.fit(X_train, y_train.astype(int))
            linear_score = roc_auc_score(y_valid.astype(int), linear.predict_proba(X_valid)[:, 1])
            tree_score = roc_auc_score(y_valid.astype(int), tree.predict_proba(X_valid)[:, 1])
            metric = "roc_auc"
        else:
            linear = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", Ridge(alpha=1.0))])
            tree = Pipeline([("impute", SimpleImputer(strategy="median")), ("model", HistGradientBoostingRegressor(max_iter=80, learning_rate=0.05, random_state=42))])
            linear.fit(X_train, y_train)
            tree.fit(X_train, y_train)
            linear_score = r2_score(y_valid, linear.predict(X_valid))
            tree_score = r2_score(y_valid, tree.predict(X_valid))
            metric = "r2"
        rows.append({
            "target": target_col,
            "feature": feature,
            "metric": metric,
            "linear_score": linear_score,
            "tree_score": tree_score,
            "tree_minus_linear": tree_score - linear_score,
            "relationship_hint": "nonlinear_or_threshold" if tree_score - linear_score > 0.03 else "mostly_linear_or_weak",
        })
    result = pd.DataFrame(rows)
    if result.empty:
        return result
    return result.sort_values(["tree_minus_linear", "tree_score"], ascending=False).head(top_n).reset_index(drop=True)

## Multivariate Feature Importance

Permutation importance trả lời: nếu tráo một feature trên validation thì metric xấu đi bao nhiêu. Đây là cách dễ đọc hơn raw tree impurity importance.

Lưu ý: feature có correlation cao với feature khác có thể bị importance thấp vì model dùng feature thay thế.

In [ ]:
def fit_importance_model(task_type: str, model_family: str, X: pd.DataFrame, y: pd.Series) -> Pipeline:
    """Input: task type, model family, X/y. Output: fitted pipeline-ready estimator before fitting."""
    if task_type == "classification":
        model = RandomForestClassifier(
            n_estimators=250,
            min_samples_leaf=8,
            max_features="sqrt",
            class_weight="balanced",
            n_jobs=-1,
            random_state=42,
        ) if model_family == "tree" else LogisticRegression(max_iter=2000, class_weight="balanced")
    else:
        model = RandomForestRegressor(
            n_estimators=250,
            min_samples_leaf=8,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42,
        ) if model_family == "tree" else Ridge(alpha=5.0)
    steps = [("preprocess", build_preprocessor(X))]
    if model_family == "linear":
        steps.append(("scale", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)


def permutation_importance_summary(
    df: pd.DataFrame,
    target_col: str,
    feature_cols: list[str],
    task_type: str = "regression",
    model_family: str = "tree",
    top_n: int = 30,
    max_rows: int = 60000,
) -> pd.DataFrame:
    """Input: dataframe, target, features, task/model type. Output: permutation-importance table on validation holdout."""
    model_df = sample_rows(df.dropna(subset=[target_col]), max_rows=max_rows)
    X = model_df[feature_cols].copy()
    y = model_df[target_col].astype(int) if task_type == "classification" else pd.to_numeric(model_df[target_col], errors="coerce")
    stratify = y if task_type == "classification" and y.nunique() == 2 else None
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42, stratify=stratify)
    model = fit_importance_model(task_type, model_family, X_train, y_train)
    model.fit(X_train, y_train)
    scoring = "roc_auc" if task_type == "classification" else "neg_mean_absolute_error"
    importance = permutation_importance(
        model,
        X_valid,
        y_valid,
        scoring=scoring,
        n_repeats=5,
        random_state=42,
        n_jobs=-1,
    )
    result = pd.DataFrame({
        "target": target_col,
        "model_family": model_family,
        "feature": feature_cols,
        "importance_mean": importance.importances_mean,
        "importance_std": importance.importances_std,
    })
    return result.sort_values("importance_mean", ascending=False).head(top_n).reset_index(drop=True)

## Short-Term Performance Targets

Targets:

- `target_next_5_pts_avg`
- `target_next_5_ast_avg`
- `target_next_5_reb_avg`

Phân tích này dùng các feature đã clean trong `performance_training_clean`. Các cột raw hiện tại (`pts`, `ast`, `reb`, `min`) được loại khỏi feature set để tránh hiểu nhầm vì chúng là box score tại game hiện tại, còn model production sau này nên ưu tiên aggregates/history.

In [ ]:
SHORT_TERM_TARGETS = {
    "points": "target_next_5_pts_avg",
    "assists": "target_next_5_ast_avg",
    "rebounds": "target_next_5_reb_avg",
}
SHORT_TERM_EXTRA_EXCLUDE = {"pts", "ast", "reb", "min"}
short_term_df = split_for_analysis(performance, "train")
short_term_features = infer_feature_columns(short_term_df, list(SHORT_TERM_TARGETS.values()), extra_exclude=SHORT_TERM_EXTRA_EXCLUDE)
print("short_term rows", short_term_df.shape)
print("short_term features", short_term_features)

In [ ]:
short_term_corr = pd.concat(
    [correlation_summary(short_term_df, target, short_term_features, top_n=20) for target in SHORT_TERM_TARGETS.values()],
    ignore_index=True,
)
short_term_mi = pd.concat(
    [mutual_information_summary(short_term_df, target, short_term_features, task_type="regression", top_n=20) for target in SHORT_TERM_TARGETS.values()],
    ignore_index=True,
)
short_term_nonlinearity = pd.concat(
    [univariate_signal_scan(short_term_df, target, short_term_features, task_type="regression", top_n=20) for target in SHORT_TERM_TARGETS.values()],
    ignore_index=True,
)

display(short_term_corr)
display(short_term_mi)
display(short_term_nonlinearity)

In [ ]:
short_term_permutation_tree = pd.concat(
    [permutation_importance_summary(short_term_df, target, short_term_features, task_type="regression", model_family="tree", top_n=20) for target in SHORT_TERM_TARGETS.values()],
    ignore_index=True,
)
short_term_permutation_linear = pd.concat(
    [permutation_importance_summary(short_term_df, target, short_term_features, task_type="regression", model_family="linear", top_n=20) for target in SHORT_TERM_TARGETS.values()],
    ignore_index=True,
)

display(short_term_permutation_tree)
display(short_term_permutation_linear)

## Long-Term Targets

Targets gồm active probability, games played, minutes per game và production per 36 ở horizon 1/2/3. H3 thường thiếu validation/test nên khi đọc output cần để ý `rows`.

In [ ]:
LONG_TERM_TARGETS = [
    "active_h1", "games_played_h1", "minutes_per_game_h1", "pts_per_36_h1", "ast_per_36_h1", "reb_per_36_h1",
    "active_h2", "games_played_h2", "minutes_per_game_h2", "pts_per_36_h2", "ast_per_36_h2", "reb_per_36_h2",
    "active_h3", "games_played_h3", "minutes_per_game_h3", "pts_per_36_h3", "ast_per_36_h3", "reb_per_36_h3",
]
long_term_df = split_for_analysis(long_term, "train")
long_term_features = infer_feature_columns(long_term_df, LONG_TERM_TARGETS)
print("long_term rows", long_term_df.shape)
print("long_term feature_count", len(long_term_features))
print(long_term_features[:50])

In [ ]:
long_term_key_targets = [
    "active_h1", "games_played_h1", "minutes_per_game_h1", "pts_per_36_h1", "ast_per_36_h1", "reb_per_36_h1",
    "active_h2", "games_played_h2", "minutes_per_game_h2",
]
long_term_corr = pd.concat(
    [correlation_summary(long_term_df, target, long_term_features, top_n=25) for target in long_term_key_targets if target in long_term_df.columns],
    ignore_index=True,
)
long_term_mi = pd.concat(
    [
        mutual_information_summary(
            long_term_df,
            target,
            long_term_features,
            task_type="classification" if target.startswith("active_") else "regression",
            top_n=25,
        )
        for target in long_term_key_targets if target in long_term_df.columns
    ],
    ignore_index=True,
)

display(long_term_corr)
display(long_term_mi)

In [ ]:
long_term_nonlinearity = pd.concat(
    [
        univariate_signal_scan(
            long_term_df,
            target,
            long_term_features,
            task_type="classification" if target.startswith("active_") else "regression",
            top_n=20,
        )
        for target in long_term_key_targets if target in long_term_df.columns
    ],
    ignore_index=True,
)

display(long_term_nonlinearity)

In [ ]:
long_term_permutation_targets = ["active_h1", "games_played_h1", "minutes_per_game_h1", "pts_per_36_h1", "ast_per_36_h1", "reb_per_36_h1"]
long_term_permutation_tree = pd.concat(
    [
        permutation_importance_summary(
            long_term_df,
            target,
            long_term_features,
            task_type="classification" if target.startswith("active_") else "regression",
            model_family="tree",
            top_n=25,
        )
        for target in long_term_permutation_targets if target in long_term_df.columns
    ],
    ignore_index=True,
)

display(long_term_permutation_tree)

## Availability Risk Targets

Tạo lại binary risk target từ `games_played_h*`:

- `low_availability_h*`: chơi `<=20` trận.
- `high_availability_h*`: chơi `>=61` trận.

Phần này giúp kiểm tra feature nào liên quan đến rủi ro mất availability.

In [ ]:
def add_availability_risk_targets(df: pd.DataFrame) -> pd.DataFrame:
    """Input: long-term dataframe. Output: copy with low/high availability binary targets."""
    result = df.copy()
    for horizon in [1, 2, 3]:
        games_col = f"games_played_h{horizon}"
        if games_col not in result.columns:
            continue
        games = pd.to_numeric(result[games_col], errors="coerce")
        result[f"low_availability_h{horizon}"] = np.where(games.notna(), (games <= 20).astype(int), np.nan)
        result[f"high_availability_h{horizon}"] = np.where(games.notna(), (games >= 61).astype(int), np.nan)
    return result


risk_df = add_availability_risk_targets(long_term_df)
risk_targets = ["low_availability_h1", "high_availability_h1", "low_availability_h2", "high_availability_h2"]
risk_features = infer_feature_columns(risk_df, LONG_TERM_TARGETS + risk_targets)

risk_corr = pd.concat(
    [correlation_summary(risk_df, target, risk_features, top_n=25) for target in risk_targets if target in risk_df.columns],
    ignore_index=True,
)
risk_mi = pd.concat(
    [mutual_information_summary(risk_df, target, risk_features, task_type="classification", top_n=25) for target in risk_targets if target in risk_df.columns],
    ignore_index=True,
)
risk_permutation = pd.concat(
    [permutation_importance_summary(risk_df, target, risk_features, task_type="classification", model_family="tree", top_n=25) for target in risk_targets if target in risk_df.columns],
    ignore_index=True,
)

display(risk_corr)
display(risk_mi)
display(risk_permutation)

## Salary Target Analysis

Salary nên ưu tiên `salary_cap_share`, vì target USD thô bị ảnh hưởng bởi salary cap inflation. Phần này giúp xem feature nào liên quan đến giá trị lương tương đối.

In [ ]:
SALARY_TARGETS = ["salary_cap_share", "target_salary_usd"]
salary_df = split_for_analysis(salary, "train")
salary_features = infer_feature_columns(salary_df, SALARY_TARGETS, extra_exclude={"salary_usd"})
print("salary rows", salary_df.shape)
print("salary feature_count", len(salary_features))

salary_corr = pd.concat(
    [correlation_summary(salary_df, target, salary_features, top_n=30) for target in SALARY_TARGETS if target in salary_df.columns],
    ignore_index=True,
)
salary_mi = pd.concat(
    [mutual_information_summary(salary_df, target, salary_features, task_type="regression", top_n=30) for target in SALARY_TARGETS if target in salary_df.columns],
    ignore_index=True,
)
salary_nonlinearity = pd.concat(
    [univariate_signal_scan(salary_df, target, salary_features, task_type="regression", top_n=25) for target in SALARY_TARGETS if target in salary_df.columns],
    ignore_index=True,
)

display(salary_corr)
display(salary_mi)
display(salary_nonlinearity)

## Feature Redundancy And Clusters

Correlation giữa các feature giúp nhận diện nhóm trùng tín hiệu. Feature rất tương quan với nhau không nhất thiết đều cần đưa vào model.

In [ ]:
def high_feature_correlation_pairs(df: pd.DataFrame, feature_cols: list[str], threshold: float = 0.85, top_n: int = 60) -> pd.DataFrame:
    """Input: dataframe and numeric features. Output: highly correlated feature pairs."""
    numeric_cols = numeric_feature_columns(df, feature_cols)
    corr = df[numeric_cols].corr(method="spearman").abs()
    pairs = []
    for i, left in enumerate(numeric_cols):
        for right in numeric_cols[i + 1:]:
            value = corr.loc[left, right]
            if pd.notna(value) and value >= threshold:
                pairs.append({"feature_a": left, "feature_b": right, "abs_spearman_corr": value})
    return pd.DataFrame(pairs).sort_values("abs_spearman_corr", ascending=False).head(top_n).reset_index(drop=True)


short_term_redundancy = high_feature_correlation_pairs(short_term_df, short_term_features, threshold=0.80)
long_term_redundancy = high_feature_correlation_pairs(long_term_df, long_term_features, threshold=0.85)
salary_redundancy = high_feature_correlation_pairs(salary_df, salary_features, threshold=0.85)

display(short_term_redundancy)
display(long_term_redundancy)
display(salary_redundancy)

## Compact Summary Tables

Các bảng summary dưới đây gom top feature từ nhiều phép đo để dễ copy output.

In [ ]:
def compact_feature_signal_summary(corr_df: pd.DataFrame, mi_df: pd.DataFrame, importance_df: pd.DataFrame | None = None) -> pd.DataFrame:
    """Input: correlation, MI, optional importance tables. Output: compact feature signal summary by target."""
    frames = []
    if not corr_df.empty:
        corr_part = corr_df[["target", "feature", "abs_spearman", "abs_pearson", "monotonic_minus_linear"]].copy()
        frames.append(corr_part)
    if not mi_df.empty:
        mi_part = mi_df[["target", "feature", "mutual_info"]].copy() if "target" in mi_df.columns else mi_df.copy()
        frames.append(mi_part)
    if importance_df is not None and not importance_df.empty:
        imp_part = importance_df[["target", "feature", "importance_mean"]].copy()
        frames.append(imp_part)
    if not frames:
        return pd.DataFrame()
    merged = frames[0]
    for frame in frames[1:]:
        merged = merged.merge(frame, on=["target", "feature"], how="outer")
    score_cols = [col for col in ["abs_spearman", "mutual_info", "importance_mean"] if col in merged.columns]
    for col in score_cols:
        max_value = merged[col].max(skipna=True)
        merged[f"rank_score_{col}"] = merged[col] / max_value if pd.notna(max_value) and max_value != 0 else np.nan
    rank_cols = [col for col in merged.columns if col.startswith("rank_score_")]
    merged["combined_signal_score"] = merged[rank_cols].mean(axis=1, skipna=True)
    return merged.sort_values(["target", "combined_signal_score"], ascending=[True, False]).reset_index(drop=True)


short_term_signal_summary = compact_feature_signal_summary(short_term_corr, short_term_mi, short_term_permutation_tree)
long_term_signal_summary = compact_feature_signal_summary(long_term_corr, long_term_mi, long_term_permutation_tree)
risk_signal_summary = compact_feature_signal_summary(risk_corr, risk_mi, risk_permutation)
salary_signal_summary = compact_feature_signal_summary(salary_corr, salary_mi, None)

display(short_term_signal_summary.groupby("target").head(15))
display(long_term_signal_summary.groupby("target").head(15))
display(risk_signal_summary.groupby("target").head(15))
display(salary_signal_summary.groupby("target").head(15))

## Notes For Reading Output

- Nếu một feature mạnh ở correlation, MI và permutation importance: rất đáng giữ.
- Nếu chỉ mạnh ở MI/tree nhưng yếu ở Pearson: khả năng có quan hệ phi tuyến, nên ưu tiên tree/boosting hoặc transform feature.
- Nếu mạnh ở correlation nhưng yếu ở permutation: có thể bị feature khác thay thế hoặc bị redundancy.
- Nếu feature mạnh bất thường và có vẻ biết tương lai, kiểm tra leakage trước khi dùng.